# 1、数据理解与处理

In [1]:
import pandas as pd

data = pd.read_csv(r'D:\8102240429\E-Commerce-Data-Analysis\data\tmall_order_report.csv')
data.head()

,订单编号,总金额,买家实际支付金额,收货地址,订单创建时间,订单付款时间,退款金额
0,1,178.8,0.0,上海,2020-02-21 00:00:00,NaN,0.0
1,2,21.0,21.0,内蒙古自治区,2020-02-20 23:59:54,2020-02-21 00:00:02,0.0
2,3,37.0,0.0,安徽省,2020-02-20 23:59:35,NaN,0.0
3,4,157.0,157.0,湖南省,2020-02-20 23:58:34,2020-02-20 23:58:44,0.0
4,5,64.8,0.0,江苏省,2020-02-20 23:57:04,2020-02-20 23:57:11,64.8


In [2]:
data.info()#数据集有28010条数据，6个字段

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28010 entries, 0 to 28009
Data columns (total 7 columns):
订单编号        28010 non-null int64
总金额         28010 non-null float64
买家实际支付金额    28010 non-null float64
收货地址        28010 non-null object
订单创建时间      28010 non-null object
订单付款时间      24087 non-null object
退款金额        28010 non-null float64
dtypes: float64(3), int64(1), object(3)
memory usage: 1.5+ MB


In [3]:
data.columns = data.columns.str.strip()#把字段名里面的空格去掉
data.columns

Index(['订单编号', '总金额', '买家实际支付金额', '收货地址', '订单创建时间', '订单付款时间', '退款金额'], dtype='object')

In [4]:
data[data.duplicated()].count()#没有重复

订单编号        0
总金额         0
买家实际支付金额    0
收货地址        0
订单创建时间      0
订单付款时间      0
退款金额        0
dtype: int64

In [5]:
data.isnull().sum()#看看有没有未付款的（付款时间为空值的）,有3923条

订单编号           0
总金额            0
买家实际支付金额       0
收货地址           0
订单创建时间         0
订单付款时间      3923
退款金额           0
dtype: int64

In [6]:
data['收货地址'] = data['收货地址'].str.replace('自治区|省|维吾尔|回族|壮族','',regex=True)
#把省份洗一下，方便可视化，这里要把正则开关打开
data['收货地址'].unique()

array(['上海', '内蒙古', '安徽', '湖南', '江苏', '浙江', '天津', '北京', '四川', '贵州', '辽宁',
       '河南', '广西', '广东', '福建', '海南', '江西', '甘肃', '河北', '黑龙江', '云南', '重庆',
       '山西', '吉林', '山东', '陕西', '湖北', '青海', '新疆', '宁夏', '西藏'], dtype=object)

In [7]:
#2、数据分析可视化

2.1 整体情况

In [8]:
result = {}
result['总订单数'] = data['订单编号'].count()
result['总订单金额'] = data['总金额'][data['订单付款时间'].notnull()].sum() #排除未付款的订单
result['总实际收入金额'] = data['买家实际支付金额'].sum() #不用排除未付款的订单，实际支付就是0
result['总退款金额'] = data['退款金额'].sum() #没有实际付款当然不产生退款，也不用排除
result['已完成订单数'] = data['订单编号'][data['订单付款时间'].notnull()].count() #排除未付款订单 
result['未付款订单数'] = data['订单编号'][data['订单付款时间'].isnull()].count()  #计算未付款订单个数
result['退款订单数'] = data['订单编号'][data['退款金额'] > 0].count() #退款的订单就是产生退款金额的（大于0）

In [9]:
result

{'总订单数': 28010,
 '总订单金额': 2474823.0700000003,
 '总实际收入金额': 1902487.15,
 '总退款金额': 572335.9199999999,
 '已完成订单数': 24087,
 '未付款订单数': 3923,
 '退款订单数': 5646}

In [10]:
from pyecharts.components import Table
from pyecharts.options import ComponentTitleOpts
table = Table()
headers = ['总订单数', '总订单金额', '已完成订单数', '总实际收入金额', '退款订单数', '总退款金额', '成交率', '退货率'] #表头
rows = [
    [
        result['总订单数'], f"{result['总订单金额']/10000:.2f} 万", result['已完成订单数'], f"{result['总实际收入金额']/10000:.2f} 万",
        result['退款订单数'], f"{result['总退款金额']/10000:.2f} 万", 
        f"{result['已完成订单数']/result['总订单数']:.2%}",
        f"{result['退款订单数']/result['已完成订单数']:.2%}",
    ]
]
table.add(headers, rows)
table.set_global_opts(
    title_opts=ComponentTitleOpts(title='整体情况')
)
table.render_notebook()

总订单数,总订单金额,已完成订单数,总实际收入金额,退款订单数,总退款金额,成交率,退货率
28010,247.48 万,24087,190.25 万,5646,57.23 万,85.99%,23.44%


2.2 地区分析

In [11]:
from pyecharts import options as opts
from pyecharts.charts import Map, Bar, Line

result2 = data[data['订单付款时间'].notnull()].groupby('收货地址').agg({'订单编号':'count'})
result21 = result2.to_dict()['订单编号']
name_map = {
    "上海":"上海市","云南":"云南省","内蒙古":"内蒙古自治区","北京":"北京市",
    "吉林":"吉林省","四川":"四川省","天津":"天津市","宁夏":"宁夏回族自治区",
    "安徽":"安徽省","山东":"山东省","山西":"山西省","广东":"广东省",
    "广西":"广西壮族自治区","新疆":"新疆维吾尔自治区","江苏":"江苏省","江西":"江西省",
    "河北":"河北省","河南":"河南省","浙江":"浙江省","海南":"海南省",
    "湖北":"湖北省","湖南":"湖南省","甘肃":"甘肃省","福建":"福建省",
    "西藏":"西藏自治区","贵州":"贵州省","辽宁":"辽宁省","重庆":"重庆市",
    "陕西":"陕西省","青海":"青海省","黑龙江":"黑龙江省","台湾":"台湾省"
}
temp_data = {name_map[k]: v for k, v in result21.items()}#这里还是得做一下映射，不然图片上是灰的
#key是拿的全称，values是拿的汇总后的订单总数
c = (
    Map()
    .add("订单量", [*temp_data.items()], "china", is_map_symbol_show=False   )
    .set_series_opts(label_opts=opts.LabelOpts(is_show=True))
    .set_global_opts(
        title_opts=opts.TitleOpts(title='天猫订单地区分布'),
        visualmap_opts=opts.VisualMapOpts(max_=1000),  #最大订单3000          
    )
)
c.render_notebook()

2.3 时间分析

In [12]:
data['订单创建时间'] = pd.to_datetime(data['订单创建时间'])
data['订单付款时间'] = pd.to_datetime(data['订单付款时间'])

In [13]:
result31 = data.groupby(data['订单创建时间'].apply(lambda x: x.strftime("%Y-%m-%d"))).agg({'订单编号':'count'}).to_dict()['订单编号']
c = (
    Line()
    .add_xaxis(list(result31.keys()))
    .add_yaxis("订单量", list(result31.values()))
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show=False),
        markpoint_opts=opts.MarkPointOpts(
            data=[
                opts.MarkPointItem(type_="max", name="最大值"),
            ]
        ),
    )
    .set_global_opts(title_opts=opts.TitleOpts(title="每日订单量走势"))
)
c.render_notebook()

从每日订单量走势来看，2 月上旬受**春节假期 + 疫情初期封控**双重影响，物流受限、消费意愿下降，订单量处于低位；2 月中下旬随着**线上消费习惯强化、部分区域物流逐步恢复**，订单量呈现**温和回升**趋势，但整体仍低于疫情前正常水平。

In [17]:
result32 = data.groupby(data['订单创建时间'].apply(lambda x: x.strftime("%H"))).agg({'订单编号':'count'}).to_dict()['订单编号']
#区别于上面，这里取的是每个小时的订单量，来分析销售量在一天内的维度上的变化，标记峰值。
x = [*result32.keys()]
y = [*result32.values()]#等同于上面把键值对list进坐标轴的操作
c = (
    Bar()
    .add_xaxis(x)
    .add_yaxis("订单量", y)
    .set_global_opts(title_opts=opts.TitleOpts(title="每小时订单量走势"))
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show=False),
        markpoint_opts=opts.MarkPointOpts(
            data=[
                opts.MarkPointItem(type_="max", name="峰值"),
                opts.MarkPointItem(name="第二峰值", coord=[x[15], y[15]], value=y[15]),
                opts.MarkPointItem(name="第三峰值", coord=[x[10], y[10]], value=y[10]),
            ]
        ),
    )
)
c.render_notebook()

一天中有三个高峰时段（10点、15点、21点），并且在21-22点有非常高的订单量。对于商家来说，商家应当在这个时间点加大客服的工作密度，并且在这时要减少商家的服务缓冲，加大对消费者的服务能力，这也是大部分电商都有夜班的原因之一。

In [18]:
s = data['订单付款时间'] - data['订单创建时间']
s[s.notnull()].apply(lambda x: x.seconds / 60 ).mean()  # 从下单到付款的平均耗时为 7.7 分钟

7.73990465119536